# ETL Silver → Gold

Popula o Star Schema no PostgreSQL a partir dos dados limpos do Silver Layer.

## 1. Imports

Bibliotecas necessárias para o ETL.


In [1]:
# Imports
import pandas as pd  # Manipulação de dados
import psycopg2  # Conexão PostgreSQL
from psycopg2.extras import execute_batch  # Inserção em lote

print("✅ Bibliotecas importadas")

✅ Bibliotecas importadas


## 2. Configuração

Parâmetros de conexão ao banco e caminho do arquivo Silver.


In [2]:
# Configuração do banco de dados
DB_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'database': 'amazon_sales',
    'user': 'postgres',
    'password': 'postgres'
}

# Caminho do arquivo Silver
SILVER_FILE = '../silver/data/amazon_products_cleaned_enhanced.csv'

print(f"📁 Silver: {SILVER_FILE}")
print(f"🔌 Database: {DB_CONFIG['database']}@{DB_CONFIG['host']}")

📁 Silver: ../silver/data/amazon_products_cleaned_enhanced.csv
🔌 Database: amazon_sales@localhost


## 3. Carregar Dados

Leitura do CSV Silver e verificação de dados com vendas.


In [3]:
# Carregar CSV do Silver
print("📥 Carregando Silver...")
df = pd.read_csv(SILVER_FILE)
print(f"✅ {len(df):,} linhas × {len(df.columns)} colunas")

# Contar registros com vendas (apenas esses vão para a fato)
vendas_count = df['units_sold_last_month'].notna().sum()
print(f"📊 Registros com vendas: {vendas_count:,}")

df.head(3)

📥 Carregando Silver...
✅ 15,938 linhas × 30 colunas
📊 Registros com vendas: 11,563


,asin,title,brand,category,rating,review_count,quality_score,final_price,original_price,price_tier,...,units_sold_last_month,revenue_last_month,date,hour,day_of_week,day_name,collected_at,bought_in_last_month_raw,is_couponed_raw,price_imputation_tier
0,1426215649,Destinations of a Lifetime: 225 of the World's...,destinations,Other,4.7,5602,93.85,20.93,NaN,Economy ($20-50),...,NaN,NaN,2025-08-21,11,3,Thursday,2025-08-21 11:43:48,List:,No Coupon,original
1,9792351833,Brother Genuine P-Touch TZe White Print on Bla...,brother,Printing,5.0,5,59.73,17.99,NaN,Budget (< $20),...,600.0,10794.0,2025-08-21,12,3,Thursday,2025-08-21 12:01:33,600+ bought in past month,No Coupon,original
2,B000001OKK,Maxell 108527 Optimally Designed Flat Packs wi...,maxell,Other,4.6,4293,91.41,7.51,NaN,Budget (< $20),...,500.0,3755.0,2025-08-21,12,3,Thursday,2025-08-21 12:12:41,500+ bought in past month,No Coupon,original


## 4. Conectar ao Banco

Estabelece conexão com o PostgreSQL.


In [4]:
# Conectar ao PostgreSQL
print("🔌 Conectando ao PostgreSQL...")
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()
print("✅ Conectado")

🔌 Conectando ao PostgreSQL...
✅ Conectado


## 5. Limpar Tabelas

Remove dados antigos para reprocessamento.


In [5]:
# Limpar todas as tabelas do Gold (TRUNCATE é mais rápido que DELETE)
print("🗑️  Limpando tabelas...")
cur.execute("TRUNCATE TABLE gold.ft_vnd, gold.dim_prdt, gold.dim_tmp, gold.dim_cat CASCADE;")
conn.commit()
print("✅ Limpas")

🗑️  Limpando tabelas...
✅ Limpas


## 6. Dimensão Tempo (dim_tmp)

Popula a dimensão temporal com atributos calculados.


In [6]:
# Popular dimensão tempo
print("\n📅 Populando dim_tmp...")
dates = df['date'].dropna().unique()
data = []

# Para cada data única, calcular atributos temporais
for date_str in dates:
    dt = pd.to_datetime(date_str)
    data.append((
        dt.date(),                          # Data
        dt.year,                            # Ano
        dt.month,                           # Mês
        dt.day,                             # Dia
        dt.dayofweek,                       # Dia da semana (0=segunda)
        dt.day_name(),                      # Nome do dia
        (dt.month - 1) // 3 + 1,           # Trimestre
        dt.isocalendar()[1],               # Semana do ano
        dt.dayofweek >= 5,                 # É fim de semana?
        dt.strftime('%Y-%m'),              # Mês-ano
        f"{dt.year}-Q{(dt.month-1)//3 + 1}" # Ano-trimestre
    ))

# Inserir em lote
execute_batch(cur, """
    INSERT INTO gold.dim_tmp 
    (data, ano, mes, dia, dia_semana, nome_dia_semana, trimestre, 
     semana_ano, eh_fim_semana, mes_ano, ano_trimestre)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (data) DO NOTHING
""", data)
conn.commit()
print(f"✅ {len(data)} datas")


📅 Populando dim_tmp...
✅ 6 datas


## 7. Dimensão Categoria (dim_cat)

Popula a dimensão de categorias com segmentos de negócio.


In [7]:
# Popular dimensão categoria
print("\n📁 Populando dim_cat...")
categorias = df['category'].dropna().unique()
data = []

# Mapeamento de categorias para segmentos de negócio
segmento_map = {
    'Audio': 'Eletrônicos', 'Camera': 'Eletrônicos', 'Mobile': 'Eletrônicos',
    'Laptop': 'Computadores', 'Storage': 'Computadores', 'Networking': 'Computadores',
    'Printing': 'Escritório', 'Power': 'Acessórios', 'Accessory': 'Acessórios'
}

# Construir registros
for cat in categorias:
    cat_str = str(cat) if pd.notna(cat) else 'Other'
    segmento = segmento_map.get(cat_str, 'Outros')
    data.append((cat_str, cat_str, segmento))

# Inserir em lote
execute_batch(cur, """
    INSERT INTO gold.dim_cat (categoria, tipo_produto, segmento)
    VALUES (%s, %s, %s)
    ON CONFLICT (categoria) DO NOTHING
""", data)
conn.commit()
print(f"✅ {len(data)} categorias")


📁 Populando dim_cat...
✅ 22 categorias


## 8. Dimensão Produto (dim_prdt)

Popula a dimensão de produtos com todos os ASINs únicos.


In [8]:
# Popular dimensão produto
print("\n📦 Populando dim_prdt...")
produtos = df[['asin', 'title', 'brand', 'category', 'price_tier',
               'best_seller_badge', 'sponsored_badge', 'is_promotable',
               'available_for_purchase']].drop_duplicates('asin')

data = []
# Construir tuplas com dados dos produtos
for _, row in produtos.iterrows():
    data.append((
        str(row['asin']),
        str(row['title'])[:500] if pd.notna(row['title']) else 'Unknown',
        str(row['brand']) if pd.notna(row['brand']) else 'Unknown',
        str(row['category']) if pd.notna(row['category']) else 'Other',
        str(row['price_tier']) if pd.notna(row['price_tier']) else 'Unknown',
        bool(row['best_seller_badge']) if pd.notna(row['best_seller_badge']) else False,
        bool(row['sponsored_badge']) if pd.notna(row['sponsored_badge']) else False,
        bool(row['is_promotable']) if pd.notna(row['is_promotable']) else False,
        bool(row['available_for_purchase']) if pd.notna(row['available_for_purchase']) else False
    ))

# Inserir em lote (ON CONFLICT atualiza campos dinâmicos)
execute_batch(cur, """
    INSERT INTO gold.dim_prdt 
    (asin, titulo, marca, categoria, faixa_preco, best_seller_badge,
     sponsored_badge, is_promotable, disponivel_compra)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (asin) DO UPDATE SET
        titulo = EXCLUDED.titulo,
        best_seller_badge = EXCLUDED.best_seller_badge,
        is_promotable = EXCLUDED.is_promotable,
        data_atualizacao = NOW()
""", data, page_size=1000)
conn.commit()
print(f"✅ {len(data)} produtos")


📦 Populando dim_prdt...
✅ 8378 produtos


## 9. Tabela Fato (ft_vnd)

Popula a tabela fato com vendas, conectando as 3 dimensões via FKs.


In [9]:
# Popular tabela fato
print("\n💰 Populando ft_vnd...")
df_facts = df[df['units_sold_last_month'].notna()].copy()
print(f"   {len(df_facts):,} registros com vendas")

# Buscar FKs das dimensões
cur.execute("SELECT asin, prdt_key FROM gold.dim_prdt")
asin_to_key = dict(cur.fetchall())

cur.execute("SELECT tmp_key, data FROM gold.dim_tmp")
date_to_key = {pd.to_datetime(d).date(): k for k, d in cur.fetchall()}

cur.execute("SELECT categoria, cat_key FROM gold.dim_cat")
cat_to_key = dict(cur.fetchall())

data = []
skipped = 0

# Construir registros da fato
for _, row in df_facts.iterrows():
    # Buscar FKs
    prdt_key = asin_to_key.get(str(row['asin']))
    tmp_key = date_to_key.get(pd.to_datetime(row['date']).date())
    cat_key = cat_to_key.get(str(row['category']) if pd.notna(row['category']) else 'Other')
    
    # Se alguma FK faltar, pular registro
    if not (prdt_key and tmp_key and cat_key):
        skipped += 1
        continue
    
    # Montar tupla com FKs + measures
    data.append((
        prdt_key, tmp_key, cat_key,
        int(row['units_sold_last_month']),
        float(row['revenue_last_month']) if pd.notna(row['revenue_last_month']) else 0.0,
        float(row['final_price']) if pd.notna(row['final_price']) else 0.0,
        float(row['rating']) if pd.notna(row['rating']) else None,
        int(row['review_count']) if pd.notna(row['review_count']) else 0,
        float(row['quality_score']) if pd.notna(row['quality_score']) else None,
        float(row['discount_pct']) if pd.notna(row['discount_pct']) else 0.0,
        int(row['hour']) if pd.notna(row['hour']) else 0,
        pd.to_datetime(row['collected_at']) if pd.notna(row['collected_at']) else None,
        str(row['price_imputation_tier']) if pd.notna(row['price_imputation_tier']) else 'unknown'
    ))

# Inserir em lote
execute_batch(cur, """
    INSERT INTO gold.ft_vnd 
    (prdt_key, tmp_key, cat_key, unidades_vendidas, receita_estimada,
     preco_final, rating, total_reviews, quality_score, percentual_desconto,
     hora_coleta, data_coleta, origem_preco)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
""", data, page_size=1000)
conn.commit()

print(f"✅ {len(data):,} registros inseridos")
if skipped > 0:
    print(f"⚠️  {skipped:,} ignorados")


💰 Populando ft_vnd...
   11,563 registros com vendas
✅ 11,563 registros inseridos


## 10. Validação

Verifica contagens e calcula métricas de negócio.


In [10]:
# Validação final
print("\n" + "="*60)
print("VALIDAÇÃO")
print("="*60)

# Contar registros em cada tabela
cur.execute("SELECT COUNT(*) FROM gold.dim_prdt")
print(f"\ndim_prdt: {cur.fetchone()[0]:,}")

cur.execute("SELECT COUNT(*) FROM gold.dim_tmp")
print(f"dim_tmp: {cur.fetchone()[0]:,}")

cur.execute("SELECT COUNT(*) FROM gold.dim_cat")
print(f"dim_cat: {cur.fetchone()[0]:,}")

cur.execute("SELECT COUNT(*) FROM gold.ft_vnd")
vnd_count = cur.fetchone()[0]
print(f"ft_vnd: {vnd_count:,}")

# Calcular métricas de negócio
if vnd_count > 0:
    cur.execute("SELECT SUM(receita_estimada), SUM(unidades_vendidas), AVG(rating) FROM gold.ft_vnd")
    revenue, units, rating = cur.fetchone()
    print(f"\nReceita: ${revenue:,.2f}")
    print(f"Unidades: {units:,}")
    print(f"Rating: {rating:.2f}" if rating else "Rating: N/A")
    print("\n✅ Gold layer pronto!")
else:
    print("\n⚠️  Fato vazia")

# Fechar conexão
conn.close()


VALIDAÇÃO

dim_prdt: 8,378
dim_tmp: 6
dim_cat: 22
ft_vnd: 11,563

Receita: $1,411,078,061.75
Unidades: 38,283,510
Rating: 4.49

✅ Gold layer pronto!
